# Rozkład na czynniki pierwsze: algorytm Shora

Komputer kwantowy (tu: symulator Qiskit Aer) rozkłada liczby typu 493 = 17 · 29.

Plan:
1. Wybieramy losowe `a`.
2. Komputer kwantowy szuka **rzędu** `r`: najmniejszego `r`, dla którego `a^r mod N = 1`.
3. Z `r` liczymy dzielnik: `gcd(a^(r/2) - 1, N)`.

Krok 2 to jedyna część kwantowa. Klasycznie jest bardzo trudny, kwantowo szybki.

In [ ]:
import math, random
from fractions import Fraction

import numpy as np
from sympy import isprime, perfect_power, factorint
from qiskit import QuantumCircuit
from qiskit.circuit.library import UnitaryGate
from qiskit.visualization import plot_histogram
from qiskit_aer import AerSimulator

sim = AerSimulator()

## Najpierw 2137

Rozkładamy 2137?

In [ ]:
isprime(2137)

2137 to liczba pierwsza, więc nie ma czego rozkładać. Bierzemy inne liczby.

## Obwód kwantowy

- **Rejestr docelowy** (`n` kubitów, `n` = liczba bitów `N`) zaczyna w stanie `1`.
- Bramka `x a mod N` mnoży go przez `a` modulo `N`. Tu budujemy ją wprost jako macierz permutacji, żeby było prosto.
- **Kubit licznika** odczytuje fazę bramki bit po bicie (estymacja fazy). Po każdym pomiarze resetujemy go i używamy ponownie, z korektą fazy zależną od wcześniejszych bitów.
- Dzięki temu wystarczy `n + 1` kubitów zamiast `3n`.

In [ ]:
def controlled_mult(a, N, n):
    """Kontrolowane mnożenie |x> -> |a*x mod N>. Kubit 0 to kontrola."""
    dim = 2 ** (n + 1)
    U = np.zeros((dim, dim), dtype=complex)
    for x in range(2**n):
        y = a * x % N if x < N else x
        U[2 * x, 2 * x] = 1
        U[2 * y + 1, 2 * x + 1] = 1
    return UnitaryGate(U, label=f"x{a} mod {N}", check_input=False)


def order_circuit(a, N):
    n = N.bit_length()
    m = 2 * n
    qc = QuantumCircuit(1 + n, m)
    qc.x(1)
    for j in range(m):
        qc.h(0)
        qc.append(controlled_mult(pow(a, 2 ** (m - 1 - j), N), N, n), range(1 + n))
        for l in range(j):
            with qc.if_test((qc.clbits[l], 1)):
                qc.p(-math.pi / 2 ** (j - l), 0)
        qc.h(0)
        qc.measure(0, j)
        qc.reset(0)
    return qc

## Przykład: N = 15, a = 7

Rząd 7 modulo 15 to 4 (7, 4, 13, 1). Wynik pomiaru `y` to w przybliżeniu `k/r · 2^m`, więc przy `m = 8` bitach wypadną tylko 0, 64, 128 i 192.

In [ ]:
qc = order_circuit(7, 15)
print("kubity:", qc.num_qubits)
counts = sim.run(qc, shots=1000).result().get_counts()
plot_histogram({int(b, 2): c for b, c in counts.items()})

## Algorytm Shora

Z `y / 2^m` ułamkiem łańcuchowym (`Fraction.limit_denominator`) odczytujemy `r`. Jeśli `r` jest nieparzyste albo `a^(r/2) = -1 mod N`, losujemy inne `a`.

In [ ]:
def find_order(a, N, shots=20):
    qc = order_circuit(a, N)
    m = qc.num_clbits
    counts = sim.run(qc, shots=shots).result().get_counts()
    for bits in sorted(counts, key=counts.get, reverse=True):
        r = Fraction(int(bits, 2), 2**m).limit_denominator(N).denominator
        if pow(a, r, N) == 1:
            return r
    return None


def shor(N):
    if isprime(N):
        print(f"{N} jest liczbą pierwszą")
        return None
    if N % 2 == 0:
        return 2, N // 2
    if perfect_power(N):
        b, _ = perfect_power(N)
        return b, N // b
    while True:
        a = random.randrange(2, N)
        if math.gcd(a, N) > 1:
            continue
        r = find_order(a, N)
        print(f"  a = {a}, rząd r = {r}")
        if r is None or r % 2 or pow(a, r // 2, N) == N - 1:
            print("  pech, losuję inne a")
            continue
        p = math.gcd(pow(a, r // 2, N) - 1, N)
        return p, N // p

## Całkiem spore liczby

Każda zajmuje kilka sekund.

In [ ]:
for N in [143, 221, 323, 437, 493]:
    print(f"{N} = {shor(N)}")

## Jeszcze większe (ok. 30 s każda)

10-bitowe liczby, 11 kubitów.

In [ ]:
for N in [899, 1007]:
    print(f"{N} = {shor(N)}")

## Granica symulatora

- Macierz bramki ma `2^(n+1) × 2^(n+1)` elementów. Każdy dodatkowy bit `N` daje 4 razy większą macierz.
- Dla 2021 = 43 · 47 (11 bitów) symulacja nie skończyła się w 10 minut.
- Prawdziwy komputer kwantowy nie trzyma macierzy, tylko wykonuje bramki. Ale do złamania RSA-2048 potrzeba około miliona fizycznych kubitów z korekcją błędów.

Dla porównania, klasyczny komputer rozkłada takie liczby od razu:

In [ ]:
factorint(2**67 - 1)